# Software Documentation Assistant: Retrieval & Evaluation Benchmark
## Week 3 | Retrieval-Augmented Generation Applications (Part C)

This notebook executes the 15-question benchmark evaluation, measuring:
1. Version Sensitivity (FastAPI 0.115 vs 0.90, Pydantic V2 vs V1)
2. API Explanation & Code Snippet Retrieval
3. Citation Correctness & Grounding
4. Unanswerable Question Abstention (Hallucination Prevention)
5. Latency and Quantitative IR Metrics (MRR, Hit Rate@4, Precision@4)

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from src.rag_pipeline import RAGPipeline
from src.evaluator import BenchmarkEvaluator, BENCHMARK_15_QUESTIONS
import pandas as pd
import matplotlib.pyplot as plt
import json

pipeline = RAGPipeline()
pipeline.build_or_load_index(force_reindex=False)
evaluator = BenchmarkEvaluator(pipeline)

### 1. Interactive Single Query Demonstration
Let's test an answerable migration query and verify citations and code.

In [ ]:
query = "How do I migrate from BaseModel.dict() to Pydantic v2 syntax?"
response = pipeline.query(query)

print(f"Query: {response['query']}")
print(f"Abstained: {response['is_abstained']}")
print(f"Confidence Score: {response['confidence_score']}")
print(f"Citations: {response['citations']}")
print(f"Latency: {response['latency']['total_ms']}ms\n")
print("--- Generated Answer ---")
print(response['answer'])

### 2. Abstention Test on Out-of-Scope Query
Testing whether the system safely refuses to hallucinate on fictitious features.

In [ ]:
out_of_scope_query = "What is the @app.quantum_accelerator() decorator in FastAPI for quantum computing speedups?"
refusal_response = pipeline.query(out_of_scope_query)

print(f"Query: {refusal_response['query']}")
print(f"Abstained: {refusal_response['is_abstained']}")
print(f"Confidence Score: {refusal_response['confidence_score']}")
print("--- Abstention Response ---")
print(refusal_response['answer'])

### 3. Run Complete 15-Question Benchmark Suite
Calculates quantitative metrics across all categories.

In [ ]:
bench_results = evaluator.run_benchmark(BENCHMARK_15_QUESTIONS)
summary = bench_results['summary']

print("\n================ BENCHMARK SUMMARY ================")
print(f"Total Questions: {summary['total_questions']}")
print(f"Hit Rate@4: {summary['hit_rate_at_k']*100:.1f}%")
print(f"MRR (Mean Reciprocal Rank): {summary['mrr']:.4f}")
print(f"Citation Correctness: {summary['citation_correctness_pct']}%")
print(f"Abstention Accuracy: {summary['abstention_accuracy_pct']}%")
print(f"Confusion Matrix: {summary['confusion_matrix']}")
print(f"Median Latency: {summary['latency_stats']['median_ms']}ms")

### 4. Results Breakdown Table

In [ ]:
df = pd.DataFrame(bench_results['results'])[['id', 'category', 'question', 'confidence_score', 'is_abstained', 'abstention_status', 'latency_ms']]
df